In [1]:
import polars as pl
from Python import identity, pitcher_features, pitcher_rolling, statcast

years = [2025]

In [2]:
print(pitcher_rolling.DEFAULT_RATE_STATS)
print(pitcher_rolling.DEFAULT_MEAN_COLS)
print(pitcher_rolling.DEFAULT_RATE_WINDOWS, pitcher_rolling.DEFAULT_MEAN_WINDOWS)

{'k_rate': ('K', 'PA'), 'bb_rate': ('BB', 'PA'), 'csw_rate': ('CSW', 'Pitches'), 'swstr_rate': ('Whiffs', 'Pitches'), 'whiff_rate': ('Whiffs', 'Swings'), 'cs_rate': ('CS', 'Pitches'), 'chase_rate': ('Chases', 'OutZone'), 'zone_rate': ('InZone', 'Pitches'), 'contact_rate': ('Contacts', 'Swings'), 'gb_rate': ('GB', 'BIP'), 'hr_rate': ('HR', 'PA')}
('ff_velo', 'ff_spinrate', 'ff_ivb', 'ff_hb', 'ff_vaa', 'si_velo', 'si_spinrate', 'si_ivb', 'si_hb', 'si_vaa', 'fc_velo', 'fc_spinrate', 'fc_ivb', 'fc_hb', 'fc_vaa', 'sl_velo', 'sl_spinrate', 'sl_ivb', 'sl_hb', 'sl_vaa', 'st_velo', 'st_spinrate', 'st_ivb', 'st_hb', 'st_vaa', 'cu_velo', 'cu_spinrate', 'cu_ivb', 'cu_hb', 'cu_vaa', 'ch_velo', 'ch_spinrate', 'ch_ivb', 'ch_hb', 'ch_vaa', 'ff_usage_vR', 'ff_usage_vL', 'si_usage_vR', 'si_usage_vL', 'fc_usage_vR', 'fc_usage_vL', 'sl_usage_vR', 'sl_usage_vL', 'st_usage_vR', 'st_usage_vL', 'cu_usage_vR', 'cu_usage_vL', 'ch_usage_vR', 'ch_usage_vL', 'extension', 'rel_x', 'rel_z', 'rel_x_sd', 'rel_z_sd', '

In [3]:
columns = ["game_year", *pitcher_features.BUILD_COLUMNS]
raw = statcast.load_statcast_years(years, columns=columns)

_, _, official_game_pks = statcast.regular_season_schedule(years[0])
statcast.validate_statcast_season(raw, years[0], official_game_pks=official_game_pks)

starts = pitcher_features.build_pitcher_starts(raw, min_batters_faced=9)
starts = identity.enrich_pitcher_names(starts, identity.load_player_map())
starts.shape

(4750, 133)

In [4]:
league_hr_fb = pitcher_features.league_hr_fb_from_pitches(raw)
starts = pitcher_features.add_fip_xfip(starts, league_hr_fb=league_hr_fb)
starts.select(["game_date", "pitcher", "pitcher_name", "PA", "K", "Outs", "FIP", "xFIP"]).head()

game_date,pitcher,pitcher_name,PA,K,Outs,FIP,xFIP
date,i64,str,u32,u32,i32,f64,f64
2025-03-18,684007,"""Shota Imanaga""",16,2,12,5.135,8.18822
2025-03-18,808967,"""Yoshinobu Yamamoto""",19,4,15,2.135,2.483939
2025-03-19,808963,"""Roki Sasaki""",14,3,9,6.135,7.298131
2025-03-19,657006,"""Justin Steele""",17,5,12,7.885,3.565871
2025-03-27,645261,"""Sandy Alcantara""",20,7,14,2.706429,3.080292


In [5]:
missing_rate_cols = {
    name: (num, den) for name, (num, den) in pitcher_rolling.DEFAULT_RATE_STATS.items()
    if num not in starts.columns or den not in starts.columns
}
missing_mean_cols = [c for c in pitcher_rolling.DEFAULT_MEAN_COLS if c not in starts.columns]

print("Rate stats skipped (missing num/den):", missing_rate_cols)
print("Mean cols skipped (missing column):", missing_mean_cols)

Rate stats skipped (missing num/den): {}
Mean cols skipped (missing column): []


In [6]:
rolled = pitcher_rolling.add_rolling_pitcher_features(starts)
rolled.shape

# Uniqueness check on the primary key -- should still be 0 duplicates after adding columns
rolled.select(["game_pk", "pitcher"]).is_duplicated().sum()

0

In [7]:
new_cols = [c for c in rolled.columns if c not in starts.columns]
print(len(new_cols), "new columns added")
new_cols

215 new columns added


['k_rate_P5',
 'k_rate_P10',
 'k_rate_P20',
 'k_rate_std',
 'bb_rate_P5',
 'bb_rate_P10',
 'bb_rate_P20',
 'bb_rate_std',
 'csw_rate_P5',
 'csw_rate_P10',
 'csw_rate_P20',
 'csw_rate_std',
 'swstr_rate_P5',
 'swstr_rate_P10',
 'swstr_rate_P20',
 'swstr_rate_std',
 'whiff_rate_P5',
 'whiff_rate_P10',
 'whiff_rate_P20',
 'whiff_rate_std',
 'cs_rate_P5',
 'cs_rate_P10',
 'cs_rate_P20',
 'cs_rate_std',
 'chase_rate_P5',
 'chase_rate_P10',
 'chase_rate_P20',
 'chase_rate_std',
 'zone_rate_P5',
 'zone_rate_P10',
 'zone_rate_P20',
 'zone_rate_std',
 'contact_rate_P5',
 'contact_rate_P10',
 'contact_rate_P20',
 'contact_rate_std',
 'gb_rate_P5',
 'gb_rate_P10',
 'gb_rate_P20',
 'gb_rate_std',
 'hr_rate_P5',
 'hr_rate_P10',
 'hr_rate_P20',
 'hr_rate_std',
 'ff_velo_P3',
 'ff_velo_P5',
 'ff_velo_P10',
 'ff_spinrate_P3',
 'ff_spinrate_P5',
 'ff_spinrate_P10',
 'ff_ivb_P3',
 'ff_ivb_P5',
 'ff_ivb_P10',
 'ff_hb_P3',
 'ff_hb_P5',
 'ff_hb_P10',
 'ff_vaa_P3',
 'ff_vaa_P5',
 'ff_vaa_P10',
 'si_velo_P3'

In [8]:
first_starts = (
    rolled.sort(["pitcher", "season", "game_date"])
    .group_by(["pitcher", "season"])
    .first()
)

# A pitcher's very first start of a season has no prior starts --
# k_rate_std and every rolling window must be null here.
first_starts.select(
    pl.col("k_rate_std").is_null().all().alias("k_rate_std_null_on_first_start"),
    pl.col("k_rate_P5").is_null().all().alias("k_rate_P5_null_on_first_start"),
)

k_rate_std_null_on_first_start,k_rate_P5_null_on_first_start
bool,bool
true,true


In [9]:
sample_pitcher = rolled["pitcher"][0]

manual_check = (
    rolled.filter(pl.col("pitcher") == sample_pitcher)
    .sort("game_date")
    .select(["game_date", "K", "PA", "k_rate_P5", "k_rate_std"])
    .head(8)
)
manual_check

# Manually verify: row N's k_rate_P5 should equal sum(K)/sum(PA) over the
# PRECEDING up-to-5 rows only, never including row N's own K/PA.

game_date,K,PA,k_rate_P5,k_rate_std
date,u32,u32,f64,f64
2025-03-29,5,22,null,null
2025-04-04,2,14,0.227273,0.227273
2025-04-09,9,26,0.194444,0.194444
2025-04-15,1,26,0.258065,0.258065
2025-04-20,6,22,0.193182,0.193182
2025-04-25,5,23,0.209091,0.209091
2025-05-01,4,25,0.207207,0.210526
2025-05-06,3,21,0.204918,0.202532


In [10]:
rolled.select(["k_rate_std", "ff_velo_P5"]).describe()

statistic,k_rate_std,ff_velo_P5
str,f64,f64
"""count""",4424.0,4295.0
"""null_count""",326.0,455.0
"""mean""",0.219174,93.933005
"""std""",0.060343,2.203071
"""min""",0.0,86.203422
"""25%""",0.178862,92.489968
"""50%""",0.21308,93.944663
"""75%""",0.255,95.390568
"""max""",0.722222,99.943703


In [11]:
if rolled["season"].n_unique() < 2:
    print("Season-boundary check skipped: load at least two seasons to test resets.")
else:
    season_boundary = (
        rolled.filter(pl.col("pitcher") == sample_pitcher)
        .sort("game_date")
        .select(["game_date", "season", "k_rate_std"])
        .with_columns(
            pl.col("season").ne(pl.col("season").shift(1))
            .fill_null(True)
            .alias("season_changed")
        )
        .filter(pl.col("season_changed"))
    )
    season_boundary

Season-boundary check skipped: load at least two seasons to test resets.
